## Key Metrics

In [1]:
import pandas as pd

sold = pd.read_pickle("..\\processed\\CRMLSSold_Clean.pkl")
listings = pd.read_pickle("..\\processed\\CRMLSListings_Clean.pkl")

### Create metrics

In [2]:
for df in [sold, listings]:
    df["Price Ratio"] = df["ClosePrice"] / df["ListPrice"]
    df["Price Per Sq Ft"] = df["ClosePrice"] / df["LivingArea"]
    df["Days on Market"] = df["DaysOnMarket"]
    df["Close to Original List Ratio"] = df["ClosePrice"] / df["OriginalListPrice"]
    df["Year"] = df["CloseDate"].dt.year
    df["Month"] = df["CloseDate"].dt.month
    df["YrMo"] = df["CloseDate"].dt.to_period("M")
    df["Listing to Contract Days"] = df["PurchaseContractDate"] - df["ListingContractDate"]
    df["Contract to Close Days"] = df["CloseDate"] - df["PurchaseContractDate"]

In [3]:
sold[["Price Ratio", "Price Per Sq Ft", "Days on Market", "Close to Original List Ratio", 
      "Year", "Month", "YrMo", "Listing to Contract Days", "Contract to Close Days"]].head()

,Price Ratio,Price Per Sq Ft,Days on Market,Close to Original List Ratio,Year,Month,YrMo,Listing to Contract Days,Contract to Close Days
0,0.813559,210.526316,777,0.480962,2024,1,2024-01,777 days,65 days
1,1.072510,412.867275,33,1.072510,2024,1,2024-01,114 days,919 days
2,1.051948,410.334347,228,1.094743,2024,1,2024-01,255 days,778 days
3,1.000000,562.098501,0,1.000000,2024,1,2024-01,0 days,48 days
4,1.000000,928.571429,66,1.000000,2024,1,2024-01,32 days,34 days


### Add School District Data
Utilize GeoJSON and geopandas to pinpoint properties within school district polygonal shapes.

In [5]:
import geopandas as gpd

districts = gpd.read_file("..\\raw\\CASchoolDistricts.geojson")

print(districts.shape)
print(districts.columns.tolist())
print(districts.head())
print(districts.crs)

(937, 54)
['OBJECTID', 'Year', 'FedID', 'CDCode', 'CDSCode', 'CountyName', 'DistrictName', 'DistrictType', 'GradeLow', 'GradeHigh', 'GradeLowCensus', 'GradeHighCensus', 'AssistStatus', 'CongressUS', 'SenateCA', 'AssemblyCA', 'UpdateNotes', 'EnrollTotal', 'EnrollCharter', 'EnrollNonCharter', 'AAcount', 'AApct', 'AIcount', 'AIpct', 'AScount', 'ASpct', 'FIcount', 'FIpct', 'HIcount', 'HIpct', 'PIcount', 'PIpct', 'WHcount', 'WHpct', 'MRcount', 'MRpct', 'NRcount', 'NRpct', 'ELcount', 'ELpct', 'FOScount', 'FOSpct', 'HOMcount', 'HOMpct', 'MIGcount', 'MIGpct', 'SWDcount', 'SWDpct', 'SEDcount', 'SEDpct', 'DistrctAreaSqMi', 'Shape__Area', 'Shape__Length', 'geometry']
   OBJECTID     Year    FedID   CDCode         CDSCode CountyName  \
0         1  2024-25  0601770  0161119  01611190000000    Alameda   
1         2  2024-25  0601860  0161127  01611270000000    Alameda   
2         3  2024-25  0604740  0161143  01611430000000    Alameda   
3         4  2024-25  0607800  0161150  01611500000000    A

Inspect district data

In [8]:
print(districts["DistrictType"].value_counts(dropna=False))

districts_subset = districts[
    ["DistrictName", "DistrictType", "CountyName", "geometry"]
].copy()

districts_subset = districts_subset.to_crs("EPSG:4326")

# Check invalid geometries
print("Invalid geometries:", (~districts_subset.geometry.is_valid).sum())

districts_subset["geometry"] = districts_subset.geometry.make_valid()
print("Invalid geometries after making valid:", (~districts_subset.geometry.is_valid).sum())

DistrictType
Elementary    516
Unified       345
High           76
Name: count, dtype: int64
Invalid geometries: 7
Invalid geometries after making valid: 0


Create geo points for property data

In [9]:
def create_property_points(df):
    valid = (
        df["Latitude"].notna() &
        df["Longitude"].notna() &
        df["Latitude"].between(32.5, 42.1) &
        df["Longitude"].between(-124.5, -114.0) &
        (df["Latitude"] != 0) &
        (df["Longitude"] != 0)
    )

    points = gpd.GeoDataFrame(
        df.loc[valid].copy(),
        geometry=gpd.points_from_xy(
            df.loc[valid, "Longitude"],
            df.loc[valid, "Latitude"]
        ),
        crs="EPSG:4326"
    )

    return points

In [10]:
sold_points = create_property_points(sold)
listings_points = create_property_points(listings)

sold_joined = gpd.sjoin(
    sold_points,
    districts_subset,
    how="left",
    predicate="within"
)

listings_joined = gpd.sjoin(
    listings_points,
    districts_subset,
    how="left",
    predicate="within"
)

Join districts and properties

In [11]:
def extract_school_districts(joined):
    result = joined.drop(
        columns=["geometry", "index_right"],
        errors="ignore"
    ).copy()

    district_types = {
        "Elementary": "ElementarySchoolDistrict",
        "High": "HighSchoolDistrict",
        "Unified": "UnifiedSchoolDistrict"
    }

    for district_type, col in district_types.items():
        values = (
            joined.loc[
                joined["DistrictType"] == district_type,
                "DistrictName"
            ]
            .groupby(level=0)
            .agg(lambda x: " | ".join(sorted(x.dropna().unique())))
        )

        result[col] = values.reindex(result.index)

    return result

In [12]:
def add_district_columns(original, joined):
    output = original.copy()

    district_cols = [
        "ElementarySchoolDistrict",
        "HighSchoolDistrict",
        "UnifiedSchoolDistrict"
    ]

    for col in district_cols:
        output[col] = pd.NA

    extracted = extract_school_districts(joined)

    for col in district_cols:
        output.loc[extracted.index, col] = extracted[col]

    return output


sold = add_district_columns(sold, sold_joined)
listings = add_district_columns(listings, listings_joined)

Separate columns for types of districts

In [13]:
district_cols = [
    "ElementarySchoolDistrict",
    "HighSchoolDistrict",
    "UnifiedSchoolDistrict"
]

for name, df in [("Sold", sold), ("Listings", listings)]:
    print(f"\n{name} district matching:")

    for col in district_cols:
        matched = df[col].notna().sum()
        print(
            f"{col}: {matched:,} / {len(df):,} "
            f"({matched / len(df) * 100:.2f}%)"
        )

print(
    sold["UnifiedSchoolDistrict"]
    .value_counts()
    .head(15)
)


Sold district matching:
ElementarySchoolDistrict: 117,774 / 480,314 (24.52%)
HighSchoolDistrict: 115,385 / 480,314 (24.02%)
UnifiedSchoolDistrict: 359,596 / 480,314 (74.87%)

Listings district matching:
ElementarySchoolDistrict: 131,899 / 644,002 (20.48%)
HighSchoolDistrict: 128,755 / 644,002 (19.99%)
UnifiedSchoolDistrict: 433,095 / 644,002 (67.25%)
UnifiedSchoolDistrict
Los Angeles Unified          48756
San Diego Unified            17415
Capistrano Unified           10022
Desert Sands Unified          9255
Palm Springs Unified          8793
Long Beach Unified            6720
Saddleback Valley Unified     6673
Oakland Unified               5603
Hemet Unified                 5492
Corona-Norco Unified          5380
Mt. Diablo Unified            4867
Riverside Unified             4725
Temecula Valley Unified       4539
Irvine Unified                4447
Poway Unified                 4272
Name: count, dtype: int64


In [14]:
sold[["ElementarySchoolDistrict", "HighSchoolDistrict", "UnifiedSchoolDistrict"]].head(10)

,ElementarySchoolDistrict,HighSchoolDistrict,UnifiedSchoolDistrict
0,San Mateo-Foster City,San Mateo Union High,NaN
1,National Elementary,Sweetwater Union High,NaN
2,National Elementary,Sweetwater Union High,NaN
3,NaN,NaN,San Luis Coastal Unified
4,NaN,NaN,Fort Bragg Unified
5,Evergreen Elementary,East Side Union High,NaN
6,NaN,NaN,San Diego Unified
7,NaN,NaN,Los Angeles Unified
8,NaN,NaN,Los Angeles Unified
9,Millbrae Elementary,San Mateo Union High,NaN


### Segment Analysis

On the sold dataset: grouped by PropertySubType, then CountyOrParish, ListOfficeName, BuyerOfficeName

In [20]:
from IPython.display import display

segment_metrics = [
    "ClosePrice",
    "ListPrice",
    "LivingArea",
    "Price Ratio",
    "Price Per Sq Ft",
    "Days on Market",
]

def summarize_segments(data, dimensions):
    segment_data = data[dimensions + segment_metrics].replace(
        [float("inf"), float("-inf")], float("nan")
    )
    grouped = segment_data.groupby(dimensions, dropna=False, sort=False)
    record_counts = grouped.size().rename("Count").to_frame()
    metrics = grouped[segment_metrics].agg(["count", "mean", "median", "std"])
    metrics.columns = [
        f"{metric}_{statistic}" for metric, statistic in metrics.columns
    ]
    return (
        record_counts.join(metrics)
        .reset_index()
        .sort_values("Count", ascending=False)
        .reset_index(drop=True)
    )

property_segment_summary = summarize_segments(
    sold, ["PropertyType", "PropertySubType"]
)
location_segment_summary = summarize_segments(
    sold, ["CountyOrParish"]
)
list_office_segment_summary = summarize_segments(
    sold, ["ListOfficeName"]
)
buyer_office_segment_summary = summarize_segments(
    sold, ["BuyerOfficeName"]
)

for title, summary in [
    ("Property type and subtype", property_segment_summary),
    ("County", location_segment_summary),
    ("Listing office", list_office_segment_summary),
    ("Buyer office", buyer_office_segment_summary),
]:
    print(f"{title}: {len(summary):,} segments")
    display(
        summary.head(20).style.format(
            "{:,.2f}", subset=summary.select_dtypes(include=["floating"]).columns
        )
    )

Property type and subtype: 21 segments


,PropertyType,PropertySubType,Count,ClosePrice_count,ClosePrice_mean,ClosePrice_median,ClosePrice_std,ListPrice_count,ListPrice_mean,ListPrice_median,ListPrice_std,LivingArea_count,LivingArea_mean,LivingArea_median,LivingArea_std,Price Ratio_count,Price Ratio_mean,Price Ratio_median,Price Ratio_std,Price Per Sq Ft_count,Price Per Sq Ft_mean,Price Per Sq Ft_median,Price Per Sq Ft_std,Days on Market_count,Days on Market_mean,Days on Market_median,Days on Market_std
0,RESIDENTIAL,SINGLEFAMILYRESIDENCE,359884,359882,"1,294,945.98","896,000.00","5,268,144.68",359884,"1,260,596.62","895,000.00","1,543,785.25",359692,"2,039.65","1,809.00","1,058.75",359882,1.61,1.00,333.79,359690,634.82,532.67,"3,994.02",359884,36.22,17.00,53.88
1,RESIDENTIAL,CONDOMINIUM,78855,78855,"866,522.23","625,000.00","8,193,818.49",78855,"771,744.60","628,000.00","672,468.59",78832,"1,262.59","1,168.00",576.09,78855,1.16,1.00,12.77,78832,696.09,562.42,"7,323.91",78855,42.52,25.00,52.09
2,RESIDENTIAL,TOWNHOUSE,28117,28117,"1,002,713.02","800,000.00","6,559,544.62",28117,"927,738.86","799,000.00","470,766.88",28112,"2,139.81","1,490.00","101,510.61",28117,1.12,1.00,10.30,28112,652.13,559.06,"4,343.99",28117,32.72,18.00,40.49
3,RESIDENTIAL,MANUFACTUREDONLAND,6202,6202,"351,122.72","320,000.00","221,426.96",6202,"358,874.56","326,900.00","224,376.66",6202,"1,439.19","1,440.00",429.70,6202,0.97,0.99,0.14,6202,246.19,225.36,140.80,6202,58.64,30.00,77.41
4,RESIDENTIAL,DUPLEX,2694,2694,"1,218,953.45","910,000.00","1,083,774.89",2694,"1,240,418.79","900,000.00","1,124,912.72",2694,"1,903.02","1,779.00",744.79,2694,0.99,1.00,0.08,2694,643.05,543.48,456.88,2694,42.64,21.00,59.58
5,RESIDENTIAL,STOCKCOOPERATIVE,1880,1880,"393,835.85","360,000.00","177,937.14",1880,"398,728.66","368,950.00","185,736.51",1880,988.42,"1,000.00","1,082.20",1880,0.99,0.99,0.04,1880,408.37,396.71,113.76,1880,38.73,21.00,49.51
6,RESIDENTIAL,nan,908,908,"970,196.90","800,000.00","693,305.50",908,"969,925.38","799,000.00","715,625.08",906,"1,732.43","1,511.00","1,156.99",908,1.00,1.00,0.07,906,"2,055.11",563.49,"31,269.80",908,44.91,20.00,73.59
7,RESIDENTIAL,CABIN,559,559,"298,157.14","250,000.00","309,192.66",559,"313,925.96","259,000.00","328,850.76",559,926.33,836.00,565.89,559,0.94,0.97,0.10,559,569.75,298.15,"5,995.42",559,77.48,45.00,97.17
8,RESIDENTIAL,TRIPLEX,414,414,"1,304,506.86","1,105,000.00","918,969.81",414,"1,349,487.69","1,150,000.00","956,811.57",414,"2,486.42","2,387.50",809.91,414,0.97,0.98,0.09,414,575.90,466.97,838.33,414,56.64,30.00,73.39
9,RESIDENTIAL,MIXEDUSE,241,241,"948,212.70","670,000.00","1,121,816.07",241,"1,024,786.48","700,000.00","1,359,483.23",240,"2,304.08","1,600.00","3,049.56",241,0.95,0.97,0.09,240,491.96,426.66,338.88,241,87.46,53.00,104.19


County: 60 segments


,CountyOrParish,Count,ClosePrice_count,ClosePrice_mean,ClosePrice_median,ClosePrice_std,ListPrice_count,ListPrice_mean,ListPrice_median,ListPrice_std,LivingArea_count,LivingArea_mean,LivingArea_median,LivingArea_std,Price Ratio_count,Price Ratio_mean,Price Ratio_median,Price Ratio_std,Price Per Sq Ft_count,Price Per Sq Ft_mean,Price Per Sq Ft_median,Price Per Sq Ft_std,Days on Market_count,Days on Market_mean,Days on Market_median,Days on Market_std
0,Los Angeles,119253,119253,"1,326,762.07","905,000.00","4,147,967.09",119253,"1,321,751.56","899,000.00","1,879,981.95",119057,"1,860.24","1,585.00","1,141.11",119253,1.02,1.00,4.17,119057,685.33,608.81,"1,852.82",119253,36.75,19.00,59.85
1,Riverside,66369,66369,"717,951.14","600,000.00","1,989,119.15",66369,"721,745.88","599,900.00","631,462.56",66366,"2,039.84","1,867.00",883.48,66369,1.01,1.00,3.80,66366,352.30,320.91,"1,326.26",66369,48.68,30.00,55.76
2,San Diego,59841,59841,"1,456,802.19","900,000.00","13,573,641.11",59841,"1,207,826.12","899,000.00","1,187,121.43",59841,"2,098.58","1,572.00","69,581.64",59841,1.38,1.00,19.43,59841,876.40,590.41,"11,695.11",59841,29.64,15.00,39.28
3,Orange,54067,54067,"1,548,917.36","1,185,000.00","3,876,371.78",54067,"1,553,519.42","1,185,000.00","1,741,546.78",54066,"1,923.25","1,681.50","1,023.05",54067,1.04,1.00,6.12,54066,771.20,674.74,"2,967.59",54067,31.66,14.00,43.09
4,San Bernardino,44638,44638,"598,046.01","534,999.50","2,294,744.20",44638,"588,416.53","535,000.00","323,181.22",44592,"1,766.93","1,621.00",870.31,44638,1.02,1.00,4.68,44592,353.71,331.93,"1,194.38",44638,46.16,25.00,60.13
5,Alameda,22702,22702,"1,305,876.51","1,135,000.00","5,167,204.75",22702,"1,188,407.08","999,999.00","683,478.77",22702,"1,736.72","1,550.00",835.62,22702,1.12,1.02,6.81,22702,769.24,699.29,"4,911.27",22702,25.91,14.00,34.00
6,Contra Costa,22026,22026,"1,149,896.07","825,000.00","4,834,153.79",22026,"1,088,669.61","799,900.00","814,291.86",22026,"1,937.80","1,728.50",963.99,22026,1.12,1.00,10.23,22026,596.92,520.54,"3,728.29",22026,28.73,14.00,36.65
7,Santa Clara,20586,20586,"1,919,000.67","1,592,000.00","1,307,240.20",20586,"1,836,716.82","1,499,000.00","1,293,422.22",20586,"1,794.32","1,600.50",863.08,20586,1.05,1.02,0.09,20586,"1,064.19",962.08,458.37,20586,22.41,10.00,32.59
8,Ventura,15095,15095,"1,077,624.05","870,000.00","853,537.24",15095,"1,089,859.39","874,900.00","867,376.60",15088,"1,949.03","1,716.00","1,030.78",15095,1.00,1.00,0.15,15088,611.34,516.46,"4,859.71",15095,43.21,28.00,50.95
9,San Mateo,8243,8243,"2,194,553.32","1,700,000.00","1,965,936.61",8243,"2,126,081.24","1,599,000.00","2,025,417.64",8243,"1,876.12","1,610.00","1,109.73",8243,1.04,1.01,0.14,8243,"1,117.78","1,051.89",403.35,8243,28.19,12.00,54.54


Listing office: 19,646 segments


,ListOfficeName,Count,ClosePrice_count,ClosePrice_mean,ClosePrice_median,ClosePrice_std,ListPrice_count,ListPrice_mean,ListPrice_median,ListPrice_std,LivingArea_count,LivingArea_mean,LivingArea_median,LivingArea_std,Price Ratio_count,Price Ratio_mean,Price Ratio_median,Price Ratio_std,Price Per Sq Ft_count,Price Per Sq Ft_mean,Price Per Sq Ft_median,Price Per Sq Ft_std,Days on Market_count,Days on Market_mean,Days on Market_median,Days on Market_std
0,Compass,34020,34020,"1,846,986.96","1,350,000.00","3,402,420.29",34020,"1,821,943.43","1,299,000.00","2,117,384.37",33953,"2,074.08","1,780.00","1,214.59",34020,1.05,1.00,5.76,33953,865.02,754.55,"2,556.57",34020,30.83,14.00,42.12
1,Coldwell Banker Realty,21578,21578,"1,658,138.68","1,200,000.00","7,292,813.61",21578,"1,589,949.45","1,188,000.00","1,680,922.53",21565,"1,978.66","1,720.00","1,079.76",21578,1.10,1.00,9.30,21565,844.77,692.31,"6,583.33",21578,34.85,16.00,49.80
2,Keller Williams Realty,9411,9411,"1,033,826.77","875,000.00","648,623.75",9411,"1,027,626.29","869,990.00","654,479.94",9411,"1,872.24","1,700.00",858.27,9411,1.01,1.00,0.08,9411,568.47,540.31,234.77,9411,31.88,15.00,42.27
3,First Team Real Estate,6726,6726,"1,123,590.30","960,000.00","723,143.28",6726,"1,121,364.69","950,000.00","724,666.64",6725,"1,855.23","1,684.00",829.81,6726,1.01,1.00,0.12,6725,619.62,608.89,280.75,6726,29.65,13.00,42.36
4,Berkshire Hathaway HomeServices California Properties,6103,6103,"1,506,442.30","957,500.00","1,726,602.61",6103,"1,533,175.23","959,000.00","1,818,288.95",6088,"2,019.65","1,716.50","1,181.82",6103,0.99,0.99,0.13,6088,702.03,600.68,474.29,6103,42.08,23.00,51.06
5,Real Broker,5602,5602,"1,053,853.41","850,000.00","746,625.73",5602,"1,054,343.95","849,000.00","762,984.01",5600,"1,822.68","1,618.00",887.67,5602,1.00,1.00,0.06,5600,586.74,558.70,252.67,5602,29.96,15.00,39.70
6,eXp Realty of California Inc,5586,5586,"977,542.66","790,000.00","812,132.57",5586,"970,300.39","785,749.50","812,091.56",5586,"1,724.40","1,556.50",786.25,5586,1.19,1.00,13.37,5586,817.97,530.43,"12,603.04",5586,37.07,19.00,48.78
7,Intero Real Estate Services,5058,5058,"1,650,845.27","1,370,000.00","1,135,619.84",5058,"1,577,979.08","1,299,999.00","1,083,650.06",5058,"1,779.40","1,598.00",786.24,5058,1.04,1.01,0.10,5058,934.13,844.46,462.24,5058,25.05,12.00,36.78
8,Equity Union,4318,4318,"1,072,663.06","851,500.00","812,550.99",4318,"1,090,621.79","859,450.00","850,137.25",4317,"2,053.17","1,840.00","1,009.08",4318,0.99,0.99,0.05,4317,518.16,477.82,225.97,4318,44.87,28.00,48.14
9,"eXp Realty of California, Inc.",4169,4169,"933,369.82","760,000.00","705,178.21",4169,"929,888.39","749,999.00","726,436.09",4169,"1,790.73","1,612.00",835.34,4169,1.01,1.00,0.06,4169,537.17,490.00,268.02,4169,28.09,13.00,39.87


Buyer office: 22,469 segments


,BuyerOfficeName,Count,ClosePrice_count,ClosePrice_mean,ClosePrice_median,ClosePrice_std,ListPrice_count,ListPrice_mean,ListPrice_median,ListPrice_std,LivingArea_count,LivingArea_mean,LivingArea_median,LivingArea_std,Price Ratio_count,Price Ratio_mean,Price Ratio_median,Price Ratio_std,Price Per Sq Ft_count,Price Per Sq Ft_mean,Price Per Sq Ft_median,Price Per Sq Ft_std,Days on Market_count,Days on Market_mean,Days on Market_median,Days on Market_std
0,Compass,31798,31798,"1,819,233.40","1,337,750.00","3,831,104.66",31798,"1,788,202.01","1,299,000.00","1,880,653.18",31754,"2,054.20","1,777.00","1,158.62",31798,1.05,1.00,5.65,31754,866.79,751.53,"2,764.82",31798,32.27,15.00,44.94
1,Coldwell Banker Realty,17308,17308,"1,704,964.73","1,199,000.00","8,887,995.92",17308,"1,604,860.36","1,189,000.00","1,768,650.92",17300,"1,987.33","1,731.00","1,083.46",17308,1.13,1.00,10.82,17300,860.50,688.65,"6,668.07",17308,35.01,16.00,50.80
2,NONMEMBER MRML,10745,10745,"634,726.27","500,000.00","646,737.19",10745,"640,967.96","499,990.00","685,069.90",10745,"1,798.01","1,638.00",821.20,10745,1.00,1.00,0.23,10745,358.51,289.60,289.78,10745,48.11,26.00,62.99
3,nan,7505,7503,"1,314,930.98","1,083,000.00","1,226,344.73",7505,"1,274,679.92","999,888.00","1,269,927.48",7505,"1,726.80","1,541.00",918.25,7503,1.03,1.00,0.16,7503,848.09,725.45,"6,813.73",7505,32.59,15.00,45.55
4,Keller Williams Realty,7410,7410,"1,239,134.82","840,000.00","12,462,661.45",7410,"1,034,647.34","839,000.00","782,681.19",7406,"1,861.90","1,668.00",890.87,7410,1.28,1.00,16.94,7406,762.65,534.30,"12,526.91",7410,34.81,17.00,45.42
5,Real Broker,7282,7282,"1,021,383.89","805,000.00","1,697,419.57",7282,"1,006,052.63","799,000.00","756,312.49",7279,"1,771.26","1,580.00",863.80,7282,1.02,1.00,1.18,7279,585.80,543.60,842.29,7282,33.26,18.00,43.04
6,eXp Realty of California Inc,6127,6127,"1,048,798.39","835,000.00","855,741.31",6127,"1,048,420.06","829,500.00","886,125.34",6126,"1,729.96","1,535.00",840.36,6127,1.17,1.00,12.76,6126,617.89,569.96,318.00,6127,37.20,19.00,47.76
7,First Team Real Estate,6097,6097,"1,092,404.98","900,000.00","891,211.57",6097,"1,088,736.82","899,000.00","898,403.10",6097,"1,808.88","1,628.00",832.37,6097,1.01,1.00,0.21,6097,614.64,598.51,315.12,6097,31.06,14.00,42.56
8,"eXp Realty of California, Inc.",5436,5436,"972,655.05","730,500.00","6,463,497.70",5436,"886,605.16","725,000.00","664,032.73",5436,"1,727.09","1,553.00",830.30,5436,1.20,1.00,14.30,5436,615.33,494.51,"6,099.09",5436,34.15,18.00,43.76
9,Berkshire Hathaway HomeServices California Properties,4741,4741,"1,516,360.56","988,888.00","1,842,577.01",4741,"1,545,089.48","995,000.00","2,030,175.65",4735,"2,025.86","1,732.00","1,250.62",4741,1.00,1.00,0.14,4735,707.77,612.47,424.19,4741,39.95,21.00,51.02


### Save datasets

In [21]:
sold.to_csv("..\\processed\\CRMLSSold_Engineered.csv")
listings.to_csv("..\\processed\\CRMLSListings_Engineered.csv")

# Save to pickle files to preserve the DataFrame structure
sold.to_pickle("..\\processed\\CRMLSSold_Engineered.pkl")
listings.to_pickle("..\\processed\\CRMLSListings_Engineered.pkl")